# 06 — Chunking strategies and model token budgets
**Goal:** Compare fixed-size and structure-preserving chunks and discover why boundaries matter.

**Must:** a retrieval unit that fits the selected embedding model; source provenance.
**Trade-off:** small chunks give precision but lose context; large chunks keep context but may dilute matching or exceed budgets. Overlap increases index size and duplicates content.

This exercise uses character limits to visualize boundaries. Real deployment must count *embedding-model tokens* rather than assume characters = tokens.

In [ ]:
from pathlib import Path
DATA = Path('../data') if Path('../data').exists() else Path('data')
text = (DATA / 'jes2_manual.md').read_text(encoding='utf-8')

def fixed_chunks(s, size=120, overlap=20):
    assert 0 <= overlap < size
    step = size - overlap
    return [s[i:i+size] for i in range(0, len(s), step)]

fixed = fixed_chunks(text)
for i, chunk in enumerate(fixed[:4], 1):
    print(f'FIXED {i}: {chunk!r}\n')

In [ ]:
def sections_from_markdown(s):
    result = []
    title = 'Preamble'
    lines = []
    for line in s.splitlines():
        if line.startswith('#'):
            if lines:
                result.append({'section': title, 'text': '\n'.join(lines).strip()})
            title = line.lstrip('#').strip()
            lines = [line]
        else:
            lines.append(line)
    if lines:
        result.append({'section': title, 'text': '\n'.join(lines).strip()})
    return result

sections = sections_from_markdown(text)
for item in sections:
    print(item['section'], '| characters:', len(item['text']))
print('Structural chunk count:', len(sections))

### Optional token count (when `transformers` is installed)
Different embedding models have different tokenizers and supported lengths. Align splitting to the *selected embedding model* later.

In [ ]:
# OPTIONAL; downloads model/tokenizer files when run for the first time
# from transformers import AutoTokenizer
# tok = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
# for chunk in fixed[:3]:
#     print(len(tok.encode(chunk, add_special_tokens=True)))

### Exercise
1. Identify a fixed chunk that splits a command, table, or paragraph.
2. Compare chunk counts and section information.
3. How would you split an oversized section while preserving its heading?

**Expected:** fixed chunks can break structure; section-aware chunks preserve headings but may become too long. Advanced algorithms combine the two.